# QNN Framework for Two-Stage Stochastic Optimization

**Paper:** [A Quantile Neural Network Framework for Two-stage Stochastic Optimization](https://arxiv.org/html/2403.11707v1)  
**Authors:** Antonio Alcántara, Carlos Ruiz, Calvin Tsay

This notebook is the single-entry-point orchestrator for the full experimental pipeline, replacing individual CLI scripts with an interactive, reproducible workflow. It covers:

1. **Setup & Configuration** – problem parameters, seeds, experiment scope
2. **Data Generation** (Algorithm 1) – fast single-scenario dataset construction
3. **Surrogate Training** – QNN and IQNN with pinball loss + early stopping
4. **Quantile Analysis** – crossing rates, pinball loss vs. dataset size
5. **Risk-Neutral Benchmark** – QNN / IQNN vs. SAA on CFLP-10-10
6. **Risk-Averse Benchmark (CVaR)** – same models, adjustable α and λ
7. **Dataset Size Impact** – validation loss across training set sizes
8. **Results Summary** – side-by-side comparison tables and charts

--- 
## 0 · Environment & Imports

In [ ]:
import time
import warnings
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from dotenv import load_dotenv
from torch.utils.data import DataLoader, TensorDataset

from qnn_stoch_opt.case_studies import (
    CFLPEvaluator,
    generate_cflp_demand_scenarios,
    generate_cflp_instance,
)
from qnn_stoch_opt.models.iqnn import IncrementalQuantileNeuralNetwork as IQNN
from qnn_stoch_opt.models.qnn import QuantileNeuralNetwork
from qnn_stoch_opt.models.trainer import train_model
from qnn_stoch_opt.optimization import SurrogateOptimizer, VarType

load_dotenv()
warnings.filterwarnings("ignore")

sns.set_theme(style="darkgrid", palette="muted", font_scale=1.1)
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
COLORS = {"QNN": "#4C72B0", "IQNN": "#DD8452", "SAA": "#55A868"}

print(f"PyTorch  : {torch.__version__}")
print(f"NumPy    : {np.__version__}")
print(f"Seaborn  : {sns.__version__}")
print(f"Device   : {'cuda' if torch.cuda.is_available() else 'cpu'}")

--- 
## 1 · Global Configuration

All experiment parameters live here. Change them to reproduce any variant.

In [ ]:
# ── Problem instance ────────────────────────────────────────────────────────
N_FACILITIES: int = 10  # CFLP facilities  (n)
N_CUSTOMERS: int = 10  # CFLP customers   (m)
INSTANCE_SEED: int = 42

# ── Dataset ─────────────────────────────────────────────────────────────────
NUM_TRAIN_SAMPLES: int = 10_000  # Total samples to generate (Algorithm 1)
VAL_FRACTION: float = 0.20  # Held-out validation fraction
DATA_SEED: int = 42

# ── Model architecture ───────────────────────────────────────────────────────
HIDDEN_DIMS: List[int] = [64, 64]
NUM_QUANTILES: int = 50
QUANTILES: torch.Tensor = torch.linspace(0.01, 0.99, NUM_QUANTILES)

# ── Training ─────────────────────────────────────────────────────────────────
EPOCHS: int = 100
LR: float = 1e-3
BATCH_SIZE: int = 64
PATIENCE: int = 10

# ── Optimization ─────────────────────────────────────────────────────────────
DELTA_QNN: float = 100.0  # Δ tolerance (Algorithm 2; paper best: 100)
ALPHA: float = 0.90  # CVaR confidence level
LAMBDA_RISK: float = 1.0  # CVaR weight λ in mean-risk objective

# ── Evaluation ───────────────────────────────────────────────────────────────
N_TEST_SCENARIOS: int = 500
TEST_SEED: int = 999

print("Configuration loaded ✓")
print(f"  Problem     : CFLP-{N_FACILITIES}-{N_CUSTOMERS}")
print(f"  Train samples: {NUM_TRAIN_SAMPLES:,}")
print(
    f"  Architecture : Linear({N_FACILITIES}) → "
    + " → ".join(f"ReLU({h})" for h in HIDDEN_DIMS)
    + f" → Linear({NUM_QUANTILES})"
)
print(f"  CVaR α={ALPHA}, λ={LAMBDA_RISK},  Δ={DELTA_QNN}")

---
## 2 · Problem Instance & Data Generation

Implements **Algorithm 1** from the paper:
1. Sample random feasible first-stage decisions $X_i$
2. Draw **one** demand scenario $\xi_i$ per decision
3. Solve the single-scenario second-stage LP → store $(X_i, v_i)$

> 💡 **Why single scenarios?**  This avoids solving expensive SAA models during data generation, making the approach scalable.

In [ ]:
# ── 2a. Generate CFLP instance ──────────────────────────────────────────────
f_costs, assignment_costs, capacities = generate_cflp_instance(
    N_FACILITIES, N_CUSTOMERS, seed=INSTANCE_SEED
)
evaluator = CFLPEvaluator(N_FACILITIES, N_CUSTOMERS, capacities, assignment_costs)

print(f"CFLP-{N_FACILITIES}-{N_CUSTOMERS} instance generated")
print(f"  First-stage costs f  : [{f_costs.min():.1f}, {f_costs.max():.1f}]")
print(f"  Capacities C         : [{capacities.min():.1f}, {capacities.max():.1f}]")
print(
    "  Assignment costs     : "
    f"[{assignment_costs.min():.1f}, {assignment_costs.max():.1f}]"
)

In [ ]:
# ── 2b. Algorithm 1 – Dataset construction ──────────────────────────────────
def generate_dataset(
    n: int,
    m: int,
    num_samples: int,
    evaluator: CFLPEvaluator,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray]:
    """Algorithm 1: fast single-scenario dataset generation."""
    X_data: List[np.ndarray] = []
    y_data: List[float] = []

    for _ in range(num_samples):
        # Random feasible binary decision (at least one facility open)
        while True:
            x_i = rng.integers(0, 2, size=n)
            if x_i.sum() > 0:
                break

        # Draw a single scenario
        scenario_i = generate_cflp_demand_scenarios(
            m, 1, seed=int(rng.integers(0, 1_000_000))
        )[0]

        # Second-stage evaluation (single-scenario LP)
        v_i = evaluator.evaluate(x_i, scenario_i)

        if v_i != float("inf"):
            X_data.append(x_i)
            y_data.append(v_i)

    return np.array(X_data, dtype=np.float32), np.array(y_data, dtype=np.float32)


rng = np.random.default_rng(DATA_SEED)

print(f"Generating {NUM_TRAIN_SAMPLES:,} training samples ...")
t0 = time.time()
X_data, y_data = generate_dataset(
    N_FACILITIES, N_CUSTOMERS, NUM_TRAIN_SAMPLES, evaluator, rng
)
gen_time = time.time() - t0

print(f"  Generated {len(X_data):,} feasible samples in {gen_time:.1f}s")
print(f"  Second-stage cost  : mean={y_data.mean():.1f},  std={y_data.std():.1f}")
print(f"  Cost range         : [{y_data.min():.1f}, {y_data.max():.1f}]")

# ── 2c. Train / Validation split ────────────────────────────────────────────
val_size = int(len(X_data) * VAL_FRACTION)
train_size = len(X_data) - val_size

train_dataset = TensorDataset(
    torch.tensor(X_data[:train_size]),
    torch.tensor(y_data[:train_size]).unsqueeze(1),
)
val_dataset = TensorDataset(
    torch.tensor(X_data[-val_size:]),
    torch.tensor(y_data[-val_size:]).unsqueeze(1),
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nDataset split: {train_size:,} train | {val_size:,} validation")

In [ ]:
# ── 2d. Second-stage cost distribution ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(y_data, bins=50, color="#4C72B0", edgecolor="white", linewidth=0.4)
axes[0].set_xlabel("Second-stage cost $v_i$")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Second-stage Costs")
axes[0].axvline(
    np.median(y_data),
    color="#DD8452",
    lw=2,
    linestyle="--",
    label=f"Median={np.median(y_data):.0f}",
)
axes[0].legend()

# Facility opening frequency across the dataset
open_freq = X_data.mean(axis=0)
axes[1].bar(
    range(1, N_FACILITIES + 1),
    open_freq,
    color="#55A868",
    edgecolor="white",
    linewidth=0.4,
)
axes[1].set_xlabel("Facility index")
axes[1].set_ylabel("Fraction of decisions opening facility")
axes[1].set_title("Facility Opening Frequency in Training Data")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.suptitle(
    f"CFLP-{N_FACILITIES}-{N_CUSTOMERS}  |  N={len(X_data):,} samples",
    y=1.02,
    fontsize=13,
)
plt.show()

---
## 3 · Surrogate Training

We train two architectures:
- **QNN** – standard quantile network; linear output; may exhibit quantile crossing
- **IQNN** – incremental QNN; ReLU increments + cumsum output; crossing-free by construction

In [ ]:
# ── 3a. Train QNN ────────────────────────────────────────────────────────────
print("Training QNN ...")
qnn = QuantileNeuralNetwork(
    input_dim=N_FACILITIES,
    hidden_dims=HIDDEN_DIMS,
    num_quantiles=NUM_QUANTILES,
)
t0 = time.time()
qnn, qnn_val_loss = train_model(
    qnn,
    train_loader,
    val_loader,
    QUANTILES,
    epochs=EPOCHS,
    lr=LR,
    patience=PATIENCE,
)
qnn_train_time = time.time() - t0
print(f"  Val pinball loss = {qnn_val_loss:.4f}  ({qnn_train_time:.1f}s)")

# ── 3b. Train IQNN ───────────────────────────────────────────────────────────
print("Training IQNN ...")
iqnn = IQNN(
    input_dim=N_FACILITIES,
    hidden_dims=HIDDEN_DIMS,
    num_quantiles=NUM_QUANTILES,
)
t0 = time.time()
iqnn, iqnn_val_loss = train_model(
    iqnn,
    train_loader,
    val_loader,
    QUANTILES,
    epochs=EPOCHS,
    lr=LR,
    patience=PATIENCE,
)
iqnn_train_time = time.time() - t0
print(f"  Val pinball loss = {iqnn_val_loss:.4f}  ({iqnn_train_time:.1f}s)")

---
## 4 · Quantile Analysis

### 4a. Crossing Rate Analysis
The **quantile crossing** phenomenon occurs when QNN predicts $\hat{q}_{k+1} < \hat{q}_k$, violating monotonicity. IQNN eliminates this by design.

In [ ]:
# ── 4a. Quantile crossing analysis ──────────────────────────────────────────
X_val_tensor = torch.tensor(X_data[-val_size:])


def crossing_stats(model: torch.nn.Module, X: torch.Tensor) -> Dict[str, float]:
    """Compute quantile crossing statistics on a validation set."""
    model.eval()
    with torch.no_grad():
        preds = model(X).numpy()  # (N, Q)

    diffs = np.diff(preds, axis=1)  # (N, Q-1)
    crossings = diffs < 0

    n_crossings_per_sample = crossings.sum(axis=1)  # (N,)
    crossing_magnitudes = (
        np.abs(diffs[crossings]) if crossings.any() else np.array([0.0])
    )

    return {
        "mean_crossings_per_sample": float(n_crossings_per_sample.mean()),
        "max_crossings_per_sample": int(n_crossings_per_sample.max()),
        "pct_samples_with_crossing": float((n_crossings_per_sample > 0).mean() * 100),
        "mean_crossing_magnitude": float(crossing_magnitudes.mean()),
        "max_crossing_magnitude": float(crossing_magnitudes.max()),
        "predictions": preds,
    }


qnn_crossing = crossing_stats(qnn, X_val_tensor)
iqnn_crossing = crossing_stats(iqnn, X_val_tensor)

crossing_df = pd.DataFrame(
    {
        "Metric": [
            "Mean crossings / sample",
            "Max crossings / sample",
            "% samples with crossing",
            "Mean crossing magnitude",
            "Max crossing magnitude",
        ],
        "QNN": [
            qnn_crossing[k]
            for k in [
                "mean_crossings_per_sample",
                "max_crossings_per_sample",
                "pct_samples_with_crossing",
                "mean_crossing_magnitude",
                "max_crossing_magnitude",
            ]
        ],
        "IQNN": [
            iqnn_crossing[k]
            for k in [
                "mean_crossings_per_sample",
                "max_crossings_per_sample",
                "pct_samples_with_crossing",
                "mean_crossing_magnitude",
                "max_crossing_magnitude",
            ]
        ],
    }
)
crossing_df = crossing_df.set_index("Metric")
crossing_df["QNN"] = crossing_df["QNN"].apply(lambda x: f"{x:.3f}")
crossing_df["IQNN"] = crossing_df["IQNN"].apply(lambda x: f"{x:.3f}")
print("=== Quantile Crossing Analysis ===")
display(crossing_df)

In [ ]:
# ── 4b. Visualise predicted quantile curves for a few samples ───────────────
n_samples_plot = 5
sample_idx = np.random.default_rng(0).integers(0, val_size, size=n_samples_plot)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
q_levels = QUANTILES.numpy()

for ax, (model, name, crossing) in zip(
    axes,
    [
        (qnn, "QNN", qnn_crossing),
        (iqnn, "IQNN", iqnn_crossing),
    ],
):
    preds = crossing["predictions"]
    for i, idx in enumerate(sample_idx):
        ax.plot(q_levels, preds[idx], alpha=0.8, lw=1.5, label=f"sample {idx}")
    ax.set_xlabel("Quantile level τ")
    ax.set_ylabel("Predicted cost")
    ax.set_title(f"{name}: Conditional Quantile Curves")
    crossing_note = f"Avg crossings: {crossing['mean_crossings_per_sample']:.2f}"
    ax.text(
        0.02,
        0.97,
        crossing_note,
        transform=ax.transAxes,
        va="top",
        fontsize=10,
        color="firebrick" if "QNN" in name else "seagreen",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.7),
    )

plt.tight_layout()
plt.suptitle(
    "Predicted Conditional Quantile Curves on Validation Samples", y=1.02, fontsize=12
)
plt.show()

---
## 5 · Risk-Neutral Benchmark

**Objective:** $\min_{x \in \mathcal{X}} \; c^\top x + \mathbb{E}[V(x,\xi)]$

The expected second-stage cost is approximated by the mean of the 50 predicted quantiles.  
We compare QNN (with Δ-tolerance correction) and IQNN against a direct SAA solve.

In [ ]:
# ── Helper: run surrogate optimization ──────────────────────────────────────
def run_surrogate_opt(
    model: torch.nn.Module,
    model_type: str,
    f_costs: np.ndarray,
    n: int,
    objective: str = "risk_neutral",
    delta: float = 0.0,
    alpha: float = 0.9,
    lambda_weight: float = 1.0,
) -> Any:
    """Embed trained model into MILP and solve."""
    x_bounds = [(0.0, 1.0)] * n
    x_vtypes = [VarType.BINARY] * n

    opt = SurrogateOptimizer(x_dim=n, x_bounds=x_bounds, var_types=x_vtypes)
    opt.embed_surrogate(model, model_type=model_type, delta=delta)

    if objective == "risk_neutral":
        opt.set_risk_neutral_objective(c=f_costs.tolist())
    elif objective == "risk_averse":
        opt.set_risk_averse_objective(c=f_costs.tolist(), alpha=alpha)
    elif objective == "mean_risk":
        opt.set_mean_risk_objective(c=f_costs.tolist(), alpha=alpha, lam=lambda_weight)
    else:
        raise ValueError(f"Unknown objective: {objective}")

    t0 = time.time()
    result = opt.optimize()
    result.solve_time_s = time.time() - t0
    return result


# ── Helper: true objective evaluation ───────────────────────────────────────
def evaluate_true_obj_risk_neutral(
    x_opt: List[float],
    f_costs: np.ndarray,
    evaluator: CFLPEvaluator,
    test_scenarios: np.ndarray,
) -> float:
    """Compute true risk-neutral objective: f^T x + E[V(x,ξ)]."""
    if not x_opt or sum(x_opt) == 0:
        return float("inf")
    costs = evaluator.evaluate_scenarios(np.array(x_opt), test_scenarios)
    valid = costs[costs != float("inf")]
    if len(valid) == 0:
        return float("inf")
    return float(np.sum(f_costs * x_opt) + np.mean(valid))


def evaluate_true_obj_risk_averse(
    x_opt: List[float],
    f_costs: np.ndarray,
    evaluator: CFLPEvaluator,
    test_scenarios: np.ndarray,
    alpha: float,
    lambda_weight: float,
) -> float:
    """Compute true risk-averse objective: f^T x + E[V] + λ·CVaR_α[V]."""
    if not x_opt or sum(x_opt) == 0:
        return float("inf")
    costs = evaluator.evaluate_scenarios(np.array(x_opt), test_scenarios)
    valid = costs[costs != float("inf")]
    if len(valid) == 0:
        return float("inf")
    first_stage = float(np.sum(f_costs * x_opt))
    var_a = float(np.quantile(valid, alpha))
    tail = valid[valid >= var_a]
    cvar = float(np.mean(tail)) if len(tail) > 0 else var_a
    return first_stage + float(np.mean(valid)) + lambda_weight * cvar


print("Helpers defined ✓")

In [ ]:
# ── 5a. Generate test scenarios ──────────────────────────────────────────────
test_scenarios = generate_cflp_demand_scenarios(
    N_CUSTOMERS, num_scenarios=N_TEST_SCENARIOS, seed=TEST_SEED
)
print(f"Test scenarios shape: {test_scenarios.shape}")

In [ ]:
# ── 5b. QNN surrogate solve (risk-neutral) ───────────────────────────────────
print("Solving Surrogate – QNN (risk-neutral) ...")
res_qnn_rn = run_surrogate_opt(
    qnn,
    "qnn",
    f_costs,
    N_FACILITIES,
    objective="risk_neutral",
    delta=DELTA_QNN,
)
true_qnn_rn = evaluate_true_obj_risk_neutral(
    res_qnn_rn.x_opt, f_costs, evaluator, test_scenarios
)
qnn_obj_str = f"{res_qnn_rn.obj_val:.2f}" if res_qnn_rn.obj_val is not None else "N/A"
print(f"  Surrogate obj : {qnn_obj_str}")
print(f"  True obj      : {true_qnn_rn:.2f}")
print(f"  Solve time    : {res_qnn_rn.solve_time_s:.3f}s")
print(f"  MIP gap       : {res_qnn_rn.mip_gap:.2%}")
print(f"  x*            : {[int(v) for v in res_qnn_rn.x_opt]}")

In [ ]:
# ── 5c. IQNN surrogate solve (risk-neutral) ──────────────────────────────────
print("Solving Surrogate – IQNN (risk-neutral) ...")
res_iqnn_rn = run_surrogate_opt(
    iqnn,
    "iqnn",
    f_costs,
    N_FACILITIES,
    objective="risk_neutral",
)
true_iqnn_rn = evaluate_true_obj_risk_neutral(
    res_iqnn_rn.x_opt, f_costs, evaluator, test_scenarios
)
iqnn_obj_str = (
    f"{res_iqnn_rn.obj_val:.2f}" if res_iqnn_rn.obj_val is not None else "N/A"
)
print(f"  Surrogate obj : {iqnn_obj_str}")
print(f"  True obj      : {true_iqnn_rn:.2f}")
print(f"  Solve time    : {res_iqnn_rn.solve_time_s:.3f}s")
print(f"  MIP gap       : {res_iqnn_rn.mip_gap:.2%}")
print(f"  x*            : {[int(v) for v in res_iqnn_rn.x_opt]}")

In [ ]:
import gurobipy as gp
from gurobipy import GRB

# ── 5d. SAA baseline ─────────────────────────────────────────────────────────
# We use a reduced scenario count here (50) to keep the demo tractable.
# The paper uses 500 scenarios and a 2-hour time limit on larger instances.
SAA_N_SCENARIOS = 50  # Increase for a fairer comparison (at the cost of time)

print(f"Solving SAA with {SAA_N_SCENARIOS} scenarios ...")

saa_scenarios = generate_cflp_demand_scenarios(
    N_CUSTOMERS, num_scenarios=SAA_N_SCENARIOS, seed=123
)

t0 = time.time()
env_saa = gp.Env(empty=True)
env_saa.setParam("OutputFlag", 0)
env_saa.start()
saa_model = gp.Model("SAA", env=env_saa)

# First stage: binary facility opening
x_saa = saa_model.addMVar(N_FACILITIES, vtype=GRB.BINARY, name="x")

# Second stage: assignment variables for each scenario
y_saa = saa_model.addMVar(
    (SAA_N_SCENARIOS, N_FACILITIES, N_CUSTOMERS), vtype=GRB.BINARY, name="y"
)

# First-stage cost
obj = gp.quicksum(f_costs[i] * x_saa[i].item() for i in range(N_FACILITIES))

# Second-stage cost averaged over scenarios
for s in range(SAA_N_SCENARIOS):
    for i in range(N_FACILITIES):
        for j in range(N_CUSTOMERS):
            obj += (
                (1.0 / SAA_N_SCENARIOS) * assignment_costs[i, j] * y_saa[s, i, j].item()
            )

saa_model.setObjective(obj, GRB.MINIMIZE)

# Constraints per scenario
for s in range(SAA_N_SCENARIOS):
    d = saa_scenarios[s]
    for i in range(N_FACILITIES):
        # Capacity
        saa_model.addConstr(
            gp.quicksum(d[j] * y_saa[s, i, j].item() for j in range(N_CUSTOMERS))
            <= capacities[i] * x_saa[i].item()
        )
    for j in range(N_CUSTOMERS):
        # Demand satisfaction
        saa_model.addConstr(
            gp.quicksum(y_saa[s, i, j].item() for i in range(N_FACILITIES)) == 1
        )
    for i in range(N_FACILITIES):
        for j in range(N_CUSTOMERS):
            saa_model.addConstr(y_saa[s, i, j].item() <= x_saa[i].item())

saa_model.optimize()
saa_solve_time = time.time() - t0

if saa_model.status == GRB.OPTIMAL:
    x_saa_opt = [float(x_saa[i].X) for i in range(N_FACILITIES)]
    saa_obj_val = float(saa_model.ObjVal)
    true_saa_rn = evaluate_true_obj_risk_neutral(
        x_saa_opt, f_costs, evaluator, test_scenarios
    )
    print(f"  SAA obj (in-sample)  : {saa_obj_val:.2f}")
    print(f"  True obj (500 scen.) : {true_saa_rn:.2f}")
    print(f"  Solve time           : {saa_solve_time:.2f}s")
    print(f"  x*                   : {[int(round(v)) for v in x_saa_opt]}")
else:
    true_saa_rn = float("inf")
    saa_solve_time = 0.0
    x_saa_opt = []
    saa_obj_val = None
    print("  SAA did not find an optimal solution.")

In [ ]:
# ── 5e. Risk-neutral results summary ────────────────────────────────────────
rn_summary = pd.DataFrame(
    [
        {
            "Method": "QNN (Δ=100)",
            "Surrogate Obj": f"{res_qnn_rn.obj_val:.2f}"
            if res_qnn_rn.obj_val is not None
            else "N/A",
            "True Obj (500 scenarios)": f"{true_qnn_rn:.2f}",
            "MIP Gap": f"{res_qnn_rn.mip_gap:.2%}",
            "Solve Time (s)": f"{res_qnn_rn.solve_time_s:.3f}",
        },
        {
            "Method": "IQNN",
            "Surrogate Obj": f"{res_iqnn_rn.obj_val:.2f}"
            if res_iqnn_rn.obj_val is not None
            else "N/A",
            "True Obj (500 scenarios)": f"{true_iqnn_rn:.2f}",
            "MIP Gap": f"{res_iqnn_rn.mip_gap:.2%}",
            "Solve Time (s)": f"{res_iqnn_rn.solve_time_s:.3f}",
        },
        {
            "Method": f"SAA ({SAA_N_SCENARIOS} scen.)",
            "Surrogate Obj": f"{saa_obj_val:.2f}" if saa_obj_val is not None else "N/A",
            "True Obj (500 scenarios)": f"{true_saa_rn:.2f}"
            if true_saa_rn != float("inf")
            else "N/A",
            "MIP Gap": "0.00%",
            "Solve Time (s)": f"{saa_solve_time:.3f}",
        },
    ]
).set_index("Method")

print(f"=== Risk-Neutral Results — CFLP-{N_FACILITIES}-{N_CUSTOMERS} ===")
display(rn_summary)

In [ ]:
# ── 5f. Visualise predicted quantile distribution at the QNN solution ────────
fig, ax = plt.subplots(figsize=(10, 4))
q_levels = QUANTILES.numpy()

for model, name, result, color in [
    (qnn, "QNN", res_qnn_rn, COLORS["QNN"]),
    (iqnn, "IQNN", res_iqnn_rn, COLORS["IQNN"]),
]:
    if not result.x_opt:
        continue
    x_tensor = torch.tensor([result.x_opt], dtype=torch.float32)
    model.eval()
    with torch.no_grad():
        q_preds = model(x_tensor).squeeze().numpy()
    ax.plot(q_levels, q_preds, label=name, color=color, lw=2)

# Overlay true cost distribution from test scenarios
if res_iqnn_rn.x_opt:
    true_costs = evaluator.evaluate_scenarios(
        np.array(res_iqnn_rn.x_opt), test_scenarios
    )
    valid_true = true_costs[true_costs != float("inf")]
    if len(valid_true) > 0:
        empirical_quantiles = np.quantile(valid_true, q_levels)
        ax.plot(
            q_levels,
            empirical_quantiles,
            "k--",
            lw=1.5,
            label="Empirical (true dist.)",
            alpha=0.8,
        )

ax.set_xlabel("Quantile level τ")
ax.set_ylabel("Second-stage cost")
ax.set_title("Predicted vs. Empirical Quantile Distribution at Optimal x*")
ax.legend()
plt.tight_layout()
plt.show()

---
## 6 · Risk-Averse Benchmark (CVaR)

**Objective:** $\min_{x \in \mathcal{X}} \; c^\top x + \mathbb{E}[V(x,\xi)] + \lambda \cdot \text{CVaR}_\alpha[V(x,\xi)]$

The same trained models (no retraining) support arbitrary $\alpha$ and $\lambda$ — a key advantage over SAA, which requires a full resolve.

In [ ]:
# ── 6a. Mean-risk solve ──────────────────────────────────────────────────────
print(f"Solving Surrogate – QNN (mean-risk, α={ALPHA}, λ={LAMBDA_RISK}) ...")
res_qnn_ra = run_surrogate_opt(
    qnn,
    "qnn",
    f_costs,
    N_FACILITIES,
    objective="mean_risk",
    delta=DELTA_QNN,
    alpha=ALPHA,
    lambda_weight=LAMBDA_RISK,
)
true_qnn_ra = evaluate_true_obj_risk_averse(
    res_qnn_ra.x_opt, f_costs, evaluator, test_scenarios, ALPHA, LAMBDA_RISK
)
print(f"  True risk-averse obj : {true_qnn_ra:.2f}")
print(f"  Solve time           : {res_qnn_ra.solve_time_s:.3f}s")

print(f"\nSolving Surrogate – IQNN (mean-risk, α={ALPHA}, λ={LAMBDA_RISK}) ...")
res_iqnn_ra = run_surrogate_opt(
    iqnn,
    "iqnn",
    f_costs,
    N_FACILITIES,
    objective="mean_risk",
    alpha=ALPHA,
    lambda_weight=LAMBDA_RISK,
)
true_iqnn_ra = evaluate_true_obj_risk_averse(
    res_iqnn_ra.x_opt, f_costs, evaluator, test_scenarios, ALPHA, LAMBDA_RISK
)
print(f"  True risk-averse obj : {true_iqnn_ra:.2f}")
print(f"  Solve time           : {res_iqnn_ra.solve_time_s:.3f}s")

In [ ]:
# ── 6b. Sensitivity to α ────────────────────────────────────────────────────
alpha_values = [0.50, 0.70, 0.80, 0.90, 0.95]
alpha_results: List[Dict[str, Any]] = []

print("Sweeping α for IQNN (mean-risk) ...")
for a in alpha_values:
    res = run_surrogate_opt(
        iqnn,
        "iqnn",
        f_costs,
        N_FACILITIES,
        objective="mean_risk",
        alpha=a,
        lambda_weight=LAMBDA_RISK,
    )
    true_obj = evaluate_true_obj_risk_averse(
        res.x_opt, f_costs, evaluator, test_scenarios, a, LAMBDA_RISK
    )
    n_open = int(sum(res.x_opt)) if res.x_opt else 0
    alpha_results.append(
        {
            "alpha": a,
            "true_obj": true_obj,
            "solve_time": res.solve_time_s,
            "n_open": n_open,
        }
    )
    print(
        f"  α={a:.2f}: true_obj={true_obj:.2f}, "
        f"time={res.solve_time_s:.3f}s, open={n_open} facilities"
    )

alpha_df = pd.DataFrame(alpha_results)

In [ ]:
# ── 6c. α sensitivity plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(
    alpha_df["alpha"],
    alpha_df["true_obj"],
    "o-",
    color=COLORS["IQNN"],
    lw=2,
    markersize=7,
)
axes[0].set_xlabel("CVaR confidence level α")
axes[0].set_ylabel("True risk-averse objective")
axes[0].set_title(f"True Obj vs. α  (IQNN, λ={LAMBDA_RISK})")

axes[1].bar(
    alpha_df["alpha"].astype(str),
    alpha_df["n_open"],
    color=COLORS["IQNN"],
    edgecolor="white",
)
axes[1].set_xlabel("CVaR confidence level α")
axes[1].set_ylabel("Number of open facilities")
axes[1].set_title("Open Facility Count vs. α")

plt.tight_layout()
plt.show()

---
## 7 · Dataset Size Impact

Reproduces **Section 4.2** of the paper: evaluating how the training dataset size affects surrogate accuracy.  
We reuse the already-generated data and retrain on progressively larger subsets.

In [ ]:
train_sizes = [500, 1000, 2000, 5000, train_size]
size_results: List[Dict[str, Any]] = []

for size in train_sizes:
    if size > train_size:
        continue
    print(f"Training with {size:,} samples ...", end=" ")

    sub_dataset = TensorDataset(
        torch.tensor(X_data[:size]),
        torch.tensor(y_data[:size]).unsqueeze(1),
    )
    sub_loader = DataLoader(sub_dataset, batch_size=BATCH_SIZE, shuffle=True)

    # QNN
    _qnn = QuantileNeuralNetwork(N_FACILITIES, HIDDEN_DIMS, NUM_QUANTILES)
    _, _qnn_loss = train_model(
        _qnn, sub_loader, val_loader, QUANTILES, epochs=EPOCHS, lr=LR, patience=PATIENCE
    )

    # IQNN
    _iqnn = IQNN(N_FACILITIES, HIDDEN_DIMS, NUM_QUANTILES)
    _, _iqnn_loss = train_model(
        _iqnn,
        sub_loader,
        val_loader,
        QUANTILES,
        epochs=EPOCHS,
        lr=LR,
        patience=PATIENCE,
    )

    size_results.append(
        {
            "train_size": size,
            "qnn_val_loss": _qnn_loss,
            "iqnn_val_loss": _iqnn_loss,
        }
    )
    print(f"QNN={_qnn_loss:.4f}  IQNN={_iqnn_loss:.4f}")

size_df = pd.DataFrame(size_results)

In [ ]:
# ── Dataset size impact plot ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    size_df["train_size"],
    size_df["qnn_val_loss"],
    "o-",
    color=COLORS["QNN"],
    label="QNN",
    lw=2,
    markersize=7,
)
ax.plot(
    size_df["train_size"],
    size_df["iqnn_val_loss"],
    "s-",
    color=COLORS["IQNN"],
    label="IQNN",
    lw=2,
    markersize=7,
)

ax.set_xscale("log")
ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax.set_xlabel("Training dataset size")
ax.set_ylabel("Validation Pinball Loss")
ax.set_title("Dataset Size Impact on Surrogate Accuracy\n(Section 4.2 of paper)")
ax.legend()

# Annotate the recommended 10k threshold
ax.axvline(10_000, color="grey", lw=1.2, linestyle="--", alpha=0.7)
ax.text(
    10_000 * 1.05,
    ax.get_ylim()[1] * 0.95,
    "Paper: 10k\nrecommended",
    fontsize=9,
    color="grey",
)

plt.tight_layout()
plt.show()

In [ ]:
# Display table
display_df = size_df.copy()
display_df["train_size"] = display_df["train_size"].apply(lambda x: f"{x:,}")
display_df.columns = ["Training Samples", "QNN Val Loss", "IQNN Val Loss"]
display_df = display_df.set_index("Training Samples")
display_df["QNN Val Loss"] = display_df["QNN Val Loss"].apply(lambda x: f"{x:.4f}")
display_df["IQNN Val Loss"] = display_df["IQNN Val Loss"].apply(lambda x: f"{x:.4f}")
print("=== Dataset Size Impact (Table 1 equivalent) ===")
display(display_df)

---
## 8 · Delta (Δ) Tolerance Sensitivity Analysis

Reproduces **Algorithm 2** evaluation: how the Δ correction for QNN quantile crossing affects solution quality and speed.

In [ ]:
delta_values = [0.0, 10.0, 50.0, 100.0, 500.0]
delta_results: List[Dict[str, Any]] = []

print("Sweeping Δ for QNN (risk-neutral) ...")
for delta in delta_values:
    res = run_surrogate_opt(
        qnn,
        "qnn",
        f_costs,
        N_FACILITIES,
        objective="risk_neutral",
        delta=delta,
    )
    true_obj = evaluate_true_obj_risk_neutral(
        res.x_opt, f_costs, evaluator, test_scenarios
    )
    delta_results.append(
        {
            "delta": delta,
            "surrogate_obj": res.obj_val,
            "true_obj": true_obj,
            "solve_time": res.solve_time_s,
            "mip_gap": res.mip_gap,
        }
    )
    print(f"  Δ={delta:6.0f}: true_obj={true_obj:.2f}  time={res.solve_time_s:.3f}s")

# Add IQNN as reference
delta_results.append(
    {
        "delta": "IQNN",
        "surrogate_obj": res_iqnn_rn.obj_val,
        "true_obj": true_iqnn_rn,
        "solve_time": res_iqnn_rn.solve_time_s,
        "mip_gap": res_iqnn_rn.mip_gap,
    }
)
delta_df = pd.DataFrame(delta_results)

In [ ]:
# ── Δ sensitivity plot ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

qnn_delta_df = delta_df[delta_df["delta"] != "IQNN"].copy()
iqnn_ref = delta_df[delta_df["delta"] == "IQNN"].iloc[0]

axes[0].plot(
    qnn_delta_df["delta"],
    qnn_delta_df["true_obj"],
    "o-",
    color=COLORS["QNN"],
    lw=2,
    markersize=7,
    label="QNN",
)
axes[0].axhline(
    iqnn_ref["true_obj"],
    color=COLORS["IQNN"],
    lw=2,
    linestyle="--",
    label="IQNN (no Δ)",
)
axes[0].set_xlabel("Δ tolerance")
axes[0].set_ylabel("True objective (500 scenarios)")
axes[0].set_title("True Objective vs. Δ Tolerance")
axes[0].legend()

axes[1].plot(
    qnn_delta_df["delta"],
    qnn_delta_df["solve_time"],
    "o-",
    color=COLORS["QNN"],
    lw=2,
    markersize=7,
    label="QNN",
)
axes[1].axhline(
    iqnn_ref["solve_time"],
    color=COLORS["IQNN"],
    lw=2,
    linestyle="--",
    label="IQNN (no Δ)",
)
axes[1].set_xlabel("Δ tolerance")
axes[1].set_ylabel("Solve time (s)")
axes[1].set_title("Solve Time vs. Δ Tolerance")
axes[1].legend()

plt.tight_layout()
plt.show()

# Display table
_disp = delta_df.copy()
_disp["Surrogate Obj"] = _disp["surrogate_obj"].apply(
    lambda x: f"{x:.2f}" if pd.notnull(x) else "N/A"
)
_disp["true_obj"] = _disp["true_obj"].apply(lambda x: f"{x:.2f}")
_disp["solve_time"] = _disp["solve_time"].apply(lambda x: f"{x:.3f}s")
_disp["mip_gap"] = _disp["mip_gap"].apply(lambda x: f"{x:.2%}")
_disp = _disp.drop(columns=["surrogate_obj"])
_disp.columns = ["Δ / Method", "True Obj", "Solve Time", "MIP Gap", "Surrogate Obj"]
_disp = _disp[["Δ / Method", "Surrogate Obj", "True Obj", "Solve Time", "MIP Gap"]]
display(_disp.set_index("Δ / Method"))

---
## 9 · Full Results Summary

In [ ]:
# ── 9a. Comprehensive comparison plot ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

methods = ["QNN", "IQNN", "SAA"]
true_rn = [true_qnn_rn, true_iqnn_rn, true_saa_rn]
times_rn = [res_qnn_rn.solve_time_s, res_iqnn_rn.solve_time_s, saa_solve_time]
colors = [COLORS["QNN"], COLORS["IQNN"], COLORS["SAA"]]

# True objective
bars = axes[0].bar(methods, true_rn, color=colors, edgecolor="white", linewidth=0.5)
axes[0].set_ylabel("True objective value")
axes[0].set_title("Risk-Neutral True Objective")
for bar, v in zip(bars, true_rn):
    if v != float("inf"):
        axes[0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 5,
            f"{v:.0f}",
            ha="center",
            va="bottom",
            fontsize=10,
        )

# Solve time (log scale)
axes[1].bar(methods, times_rn, color=colors, edgecolor="white", linewidth=0.5)
axes[1].set_yscale("log")
axes[1].set_ylabel("Solve time (s, log scale)")
axes[1].set_title("Solver Time Comparison")
for i, (m, t) in enumerate(zip(methods, times_rn)):
    axes[1].text(i, t * 1.2, f"{t:.3f}s", ha="center", va="bottom", fontsize=9)

# Risk-averse true objective
true_ra = [true_qnn_ra, true_iqnn_ra]
axes[2].bar(
    ["QNN", "IQNN"],
    true_ra,
    color=[COLORS["QNN"], COLORS["IQNN"]],
    edgecolor="white",
    linewidth=0.5,
)
axes[2].set_ylabel("True risk-averse objective")
axes[2].set_title(f"Risk-Averse (Mean-Risk, α={ALPHA}, λ={LAMBDA_RISK})")
for i, v in enumerate(true_ra):
    axes[2].text(i, v + 5, f"{v:.0f}", ha="center", va="bottom", fontsize=10)

plt.suptitle(
    f"CFLP-{N_FACILITIES}-{N_CUSTOMERS} Full Results Summary", fontsize=13, y=1.02
)
plt.tight_layout()
plt.show()

In [ ]:
# ── 9b. Printable summary table ──────────────────────────────────────────────
summary_rows = [
    {
        "Experiment": "Risk-Neutral",
        "Method": "QNN (Δ=100)",
        "Val Pinball Loss": f"{qnn_val_loss:.4f}",
        "True Obj": f"{true_qnn_rn:.2f}",
        "Solve Time (s)": f"{res_qnn_rn.solve_time_s:.3f}",
        "MIP Gap": f"{res_qnn_rn.mip_gap:.2%}",
    },
    {
        "Experiment": "Risk-Neutral",
        "Method": "IQNN",
        "Val Pinball Loss": f"{iqnn_val_loss:.4f}",
        "True Obj": f"{true_iqnn_rn:.2f}",
        "Solve Time (s)": f"{res_iqnn_rn.solve_time_s:.3f}",
        "MIP Gap": f"{res_iqnn_rn.mip_gap:.2%}",
    },
    {
        "Experiment": "Risk-Neutral",
        "Method": f"SAA ({SAA_N_SCENARIOS} scenarios)",
        "Val Pinball Loss": "—",
        "True Obj": f"{true_saa_rn:.2f}" if true_saa_rn != float("inf") else "N/A",
        "Solve Time (s)": f"{saa_solve_time:.3f}",
        "MIP Gap": "—",
    },
    {
        "Experiment": f"Risk-Averse (α={ALPHA}, λ={LAMBDA_RISK})",
        "Method": "QNN (Δ=100)",
        "Val Pinball Loss": f"{qnn_val_loss:.4f}",
        "True Obj": f"{true_qnn_ra:.2f}",
        "Solve Time (s)": f"{res_qnn_ra.solve_time_s:.3f}",
        "MIP Gap": f"{res_qnn_ra.mip_gap:.2%}",
    },
    {
        "Experiment": f"Risk-Averse (α={ALPHA}, λ={LAMBDA_RISK})",
        "Method": "IQNN",
        "Val Pinball Loss": f"{iqnn_val_loss:.4f}",
        "True Obj": f"{true_iqnn_ra:.2f}",
        "Solve Time (s)": f"{res_iqnn_ra.solve_time_s:.3f}",
        "MIP Gap": f"{res_iqnn_ra.mip_gap:.2%}",
    },
]

summary_df = pd.DataFrame(summary_rows).set_index(["Experiment", "Method"])
print(f"\n{'=' * 70}")
print(f"  FINAL RESULTS — CFLP-{N_FACILITIES}-{N_CUSTOMERS}")
print(f"{'=' * 70}")
display(summary_df)

# ── Speed-up vs. SAA ────────────────────────────────────────────────────────
if saa_solve_time > 0:
    print(f"\n  Speed-up QNN  vs SAA : {saa_solve_time / res_qnn_rn.solve_time_s:.0f}×")
    print(f"  Speed-up IQNN vs SAA : {saa_solve_time / res_iqnn_rn.solve_time_s:.0f}×")

---
## 10 · Save Trained Models & Results

Persist models and the numerical results to `results/` for reproducible reference.

In [ ]:
import json
from pathlib import Path

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

# ── Save model weights ──────────────────────────────────────────────────────
torch.save(qnn.state_dict(), results_dir / "qnn_cflp10x10.pt")
torch.save(iqnn.state_dict(), results_dir / "iqnn_cflp10x10.pt")
print("Models saved ✓")

# ── Save numerical results ──────────────────────────────────────────────────
results_payload = {
    "config": {
        "n_facilities": N_FACILITIES,
        "n_customers": N_CUSTOMERS,
        "num_train": NUM_TRAIN_SAMPLES,
        "hidden_dims": HIDDEN_DIMS,
        "num_quantiles": NUM_QUANTILES,
        "delta_qnn": DELTA_QNN,
        "alpha": ALPHA,
        "lambda_risk": LAMBDA_RISK,
    },
    "training": {
        "qnn_val_loss": qnn_val_loss,
        "iqnn_val_loss": iqnn_val_loss,
        "qnn_train_time_s": qnn_train_time,
        "iqnn_train_time_s": iqnn_train_time,
    },
    "crossing_analysis": {
        "qnn_mean_crossings": qnn_crossing["mean_crossings_per_sample"],
        "iqnn_mean_crossings": iqnn_crossing["mean_crossings_per_sample"],
    },
    "risk_neutral": {
        "qnn": {
            "true_obj": true_qnn_rn,
            "solve_time": res_qnn_rn.solve_time_s,
            "mip_gap": res_qnn_rn.mip_gap,
        },
        "iqnn": {
            "true_obj": true_iqnn_rn,
            "solve_time": res_iqnn_rn.solve_time_s,
            "mip_gap": res_iqnn_rn.mip_gap,
        },
        "saa": {
            "true_obj": true_saa_rn,
            "solve_time": saa_solve_time,
            "n_scenarios": SAA_N_SCENARIOS,
        },
    },
    "risk_averse": {
        "qnn": {"true_obj": true_qnn_ra, "solve_time": res_qnn_ra.solve_time_s},
        "iqnn": {"true_obj": true_iqnn_ra, "solve_time": res_iqnn_ra.solve_time_s},
    },
}

with open(results_dir / "experiment_results.json", "w") as f:
    json.dump(results_payload, f, indent=2)
print("Results saved to results/experiment_results.json ✓")

# ── Save CSVs for further analysis ──────────────────────────────────────────
size_df.to_csv(results_dir / "dataset_size_impact.csv", index=False)
alpha_df.to_csv(results_dir / "alpha_sensitivity.csv", index=False)
delta_df.to_csv(results_dir / "delta_sensitivity.csv", index=False)
print("CSVs saved ✓")